In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy samplomatic
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

<Accordion>
<AccordionItem title="패키지 버전">

이 페이지의 코드는 다음 요구 사항을 사용하여 개발되었습니다.
이 버전 이상을 사용하는 것을 권장합니다.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
samplomatic~=0.18.0
```
</AccordionItem>
</Accordion>

Executor Primitive는 오류 완화 워크플로를 사용자 정의할 때 더 많은 유연성을 제공하는 [지향적 실행 모델](/guides/directed-execution-model)의 일부입니다.

Executor Primitive의 입력과 출력은 [Sampler](/guides/sampler-input-output) 및 [Estimator](/guides/estimator-input-output) Primitive의 입력과 출력과 매우 다릅니다. 예를 들어, PUB 목록을 입력으로 받는 대신 Executor는 `QuantumProgramItem` 객체 목록을 포함하는 `QuantumProgram`을 받습니다. 이 컨테이너 클래스는 단순한 튜플 데이터 구조인 PUB보다 더 많은 유연성을 제공합니다.

Executor의 출력은 `QuantumProgramResult`로, 반복 가능하며 각 입력 `QuantumProgramItem`에 대해 하나의 요소를 포함합니다.

<span id="programs"></span>

## 입력: 양자 프로그램

앞서 언급했듯이, Executor Primitive에 대한 입력은 [`QuantumProgram`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/quantum-program-quantum-program)으로, [`QuantumProgramItem`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/quantum-program-quantum-program-item) 객체의 반복 가능한 컬렉션입니다. 이 객체들은 두 가지 유형이 될 수 있습니다:
- `CircuitItem`: 일반적으로 Circuit과 해당 매개변수 값(있는 경우)을 저장합니다.
- `SamplexItem`: 일반적으로 다음을 저장합니다:
    - 템플릿 Circuit
    - 런타임에 무작위화된 매개변수 집합을 생성하는 데 사용되는 samplex 객체(예: 트월링 수행 또는 노이즈 주입)
    - 원래 Circuit에 대한 매개변수 값을 포함할 수 있는 samplex에 대한 인수

이 각 항목은 Executor가 수행할 다른 작업을 나타냅니다.

### 시작하기 전에

이 페이지의 일부 코드 예제는 Samplomatic 패키지의 일부인 `samplex`를 사용합니다. 따라서 해당 코드 블록을 실행하기 전에 다음 코드 블록에 표시된 것처럼 Samplomatic을 설치해야 합니다. 자세한 내용은 [Samplomatic 문서](https://qiskit.github.io/samplomatic)를 참조하세요.

In [1]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.quantum_program import QuantumProgram
from qiskit_ibm_runtime import Executor, QiskitRuntimeService
from qiskit.circuit import Parameter, QuantumCircuit
import numpy as np
from samplomatic import build
from samplomatic.transpiler import generate_boxing_pass_manager

# Initialize an empty program
program = QuantumProgram(shots=1024)

# Initialize and transpile a 3-qubit quantum circuit with 2 parameters.
circuit = QuantumCircuit(3)
circuit.h(0)
circuit.cx(0, 1)
circuit.cx(1, 2)
circuit.rz(Parameter("theta"), 0)
circuit.rz(Parameter("phi"), 1)

# `measure_all` adds a 3-bit classical register named "meas"
circuit.measure_all()

# Choose the least busy backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Generate a preset pass manager
# This will be used to convert the abstract circuit to an
# equivalent Instruction Set Architecture (ISA) circuit.
preset_pass_manager = generate_preset_pass_manager(
    backend=backend, optimization_level=0
)

# Transpile the circuit
isa_circuit = preset_pass_manager.run(circuit)

### 예제: 두 가지 다른 작업으로 `QuantumProgram` 생성
먼저 양자 프로그램을 초기화한 다음, 다음 예제에 표시된 것처럼 `append_circuit_item` 또는 `append_samplex_item`(samplex가 있는 경우)를 사용하여 프로그램 항목을 추가합니다.

다음 셀은 `QuantumProgram`을 초기화하고 프로그램의 각 항목의 모든 구성에 대해 1024번의 샷을 실행하도록 지정합니다.

> **Note:** Sampler와 달리, `QuantumProgram`은 단일 샷 값만 받습니다. 다른 샷 값을 원하는 경우 별도의 `QuantumProgram`이 필요하며 이는 별도의 Job이 됩니다.

In [2]:
# Append the transpiled circuit and an array
# containing 10 sets of parameter values to the program
program.append_circuit_item(
    isa_circuit,
    circuit_arguments=np.random.rand(
        10, 2
    ),  # 10 sets of parameter values and 2 parameters
)

### `CircuitItem` 추가
다음으로, 백엔드의 명령 집합 아키텍처(ISA)에 따라 트랜스파일된 대상 Circuit을 `QuantumProgram`에 추가합니다. 이 Circuit에는 두 개의 매개변수가 있으므로 매개변수 값도 제공해야 합니다(이 예제에서는 10개 집합). 이 `CircuitItem`을 실행하는 것이 프로그램이 수행할 첫 번째 작업입니다.

In [3]:
# Transpile the circuit, additionally grouping gates and measurements into annotated boxes
preset_pass_manager = generate_preset_pass_manager(
    backend=backend, optimization_level=0
)

# Use the boxing pass manager to group gates
# and measurements into boxes and add
# a`Twirl` annotation.
preset_pass_manager.post_scheduling = generate_boxing_pass_manager(
    # Add gate twirling
    enable_gates=True,
    # Add measurement twirling
    enable_measures=True,
)
boxed_circuit = preset_pass_manager.run(circuit)

# Build the template circuit and the samplex.  The template circuit has parametric gates
# without fixed values and the samplex randomly generates the parameter
# values on the server side at runtime to perform twirling.
template_circuit, samplex = build(boxed_circuit)

# Determine what arguments are required by the samplex.
# Input the arguments in samplex_arguments.
print(samplex.inputs())

TensorInterface(<
  - 'parameter_values' <float64[2]>: Input parameter values to use during sampling.
>)


In [4]:
# Append the template circuit and samplex as a samplex item
program.append_samplex_item(
    template_circuit,
    samplex=samplex,
    samplex_arguments={
        # the arguments required by the samplex.sample method
        "parameter_values": np.random.rand(10, 2),
    },
    shape=(28, 10),  # 28 randomizations and 10 sets of parameter values
)

In [5]:
# Initialize an Executor with the default options
executor = Executor(mode=backend)

# Submit the job
job = executor.run(program)

# Retrieve the result
result = job.result()

## Outputs

Executor's output is a [`QuantumProgramResult`](/docs/api/qiskit-ibm-runtime/results-quantum-program-result), which is an iterable. It contains one entry per input `QuantumProgramItem` in the same order as the input items. Each of these output items is a dictionary where the keys are strings that correspond the classical registers' names in the input circuits (among others), so you no longer need to memorize these names like you did with Sampler output. The dictionary values are of type `np.ndarray`.

The result for the previous example contains these items:

### `CircuitItem` result

The first item contains the results of running the first task (a `CircuitItem`) in the program. It contains a single key, `meas`, which is the name of the classical register in the input circuit. The value of this key maps to an `np.ndarray` of shape `(parameter sets, shots, register bits)`, which is (10, 1024, 3) for the above example.

The following code illustrates how to access this information:

In [6]:
# Access the results of the classical register of task #0, a CircuitItem
result_0 = result[0]["meas"]
print(f"Result shape: {result_0.shape}")

Result shape: (10, 1024, 3)


### `SamplexItem` result

The second item contains the results of running the second task (a `SamplexItem`) in the program. This item contains multiple keys. The `meas` key, which is the name of the input circuit's classical register, maps to that register's array of results. This array has the shape `(randomizations, parameter sets, shots, classical bits)`, or (28, 10, 1024, 3) in this example. Additionally, the output contains a `measurement_flips.meas` key, which is the bit-flip corrections to undo the measurement twirling for the `meas` register.  This output shape will be (28, 10, 1, 3) for our example because only one shot is required to perform the bit-flip.

In [7]:
# Access the results of the classical register of task #1
result_1 = result[1]["meas"]
print(f"Result shape: {result_1.shape}")

# Access the bit-flip corrections
flips_1 = result[1]["measurement_flips.meas"]
print(f"Bit-flip corrections shape: {flips_1.shape}")

# Undo the bit flips via classical XOR
unflipped_result_1 = result_1 ^ flips_1

Result shape: (28, 10, 1024, 3)
Bit-flip corrections shape: (28, 10, 1, 3)


## 출력
Executor의 출력은 반복 가능한 [`QuantumProgramResult`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/results-quantum-program-result)입니다. 입력 항목과 같은 순서로 입력 `QuantumProgramItem`당 하나의 항목을 포함합니다. 이 출력 항목 각각은 딕셔너리로, 키는 입력 Circuit의 고전 레지스터 이름에 해당하는 문자열이므로(그 외에도) Sampler 출력처럼 이 이름들을 기억할 필요가 없습니다. 딕셔너리 값은 `np.ndarray` 유형입니다.

이전 예제의 결과에는 다음 항목들이 포함됩니다:

### `CircuitItem` 결과
첫 번째 항목은 프로그램에서 첫 번째 작업(`CircuitItem`)을 실행한 결과를 포함합니다. 입력 Circuit의 고전 레지스터 이름인 `meas`라는 단일 키를 포함합니다. 이 키의 값은 `(매개변수 집합, 샷, 레지스터 비트)` 모양의 `np.ndarray`에 매핑되며, 위 예제의 경우 (10, 1024, 3)입니다.

다음 코드는 이 정보에 접근하는 방법을 보여줍니다: